# Phase two: what the trained models did to their representation

Phase one says which arm wins. It does not say whether it won by aligning the
domains class by class, which is what both formulations claim. That claim lives
in the representation, so it is measured there.

This notebook trains nothing. It loads the checkpoints phase one kept — the
repetitions closest to the median, never the best — each beside the manifest that
records the exact images its bags were built from, so the space being measured is
the space that produced the accuracy.

**Two numbers are needed together and neither means anything alone.** If the
distance between the two domains' same-class centroids falls while different
classes stay apart, that is conditional alignment. If both fall, the space
collapsed — and the first number by itself would have called that a success.

Each arm is read against its own floor: the same arm with the adaptation term
switched off. An embedding has no absolute scale, so what carries information is
how it moved, not where it sits.

The projections are here to look at. The numbers are what decide.

In [1]:
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [2]:
import json

import torch

from MIL_CREDA_Benchmark import config, harness, latent

device = harness.resolve_device()
found = latent.available()
if not found:
    raise SystemExit(
        f"no checkpoints in {config.MODELS}. Run phase one first: it keeps the "
        f"repetitions closest to the median for {sorted(config.CHECKPOINTS)}."
    )
print(f"{len(found)} checkpoints on {device}")
for record in found:
    print(f"  {record['arm']:>2} {record['transfer']:<6} seed {record['seed']:<3} "
          f"target {record['targetAccuracy']:.3f}")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


24 checkpoints on mps
   A M->S   seed 0   target 0.083
   A M->U   seed 0   target 0.778
   A S->M   seed 0   target 0.722
   A S->U   seed 0   target 0.472
   A U->M   seed 0   target 0.972
   A U->S   seed 0   target 0.194
   B M->S   seed 0   target 0.222
   B M->U   seed 0   target 0.472
   B S->M   seed 0   target 0.389
   B S->U   seed 0   target 0.333
   B U->M   seed 0   target 0.833
   B U->S   seed 0   target 0.111
   D M->S   seed 0   target 0.139
   D M->U   seed 0   target 0.722
   D S->M   seed 0   target 0.528
   D S->U   seed 0   target 0.361
   D U->M   seed 0   target 0.972
   D U->S   seed 0   target 0.194
   G M->S   seed 0   target 0.194
   G M->U   seed 0   target 0.389
   G S->M   seed 0   target 0.333
   G S->U   seed 0   target 0.250
   G U->M   seed 0   target 1.000
   G U->S   seed 0   target 0.083


In [3]:
readings = [latent.analyse(record, device) for record in found]
print(f"{'arm':<4}{'transfer':<8}{'seed':>5}{'unit':>10}"
      f"{'same class':>12}{'between':>10}{'ratio':>8}{'separable':>11}")
for reading in readings:
    geometry = reading["geometry"]
    print(f"{reading['arm']:<4}{reading['transfer']:<8}{reading['seed']:>5}"
          f"{reading['unit']:>10}{geometry['crossDomainSameClass']:>12.3f}"
          f"{geometry['betweenClasses']:>10.3f}{geometry['ratio']:>8.3f}"
          f"{reading['domainSeparability']:>11.3f}")

arm transfer seed      unit  same class   between   ratio  separable
A   M->S        0  instance      27.487    22.816   1.205      0.947
A   M->U        0  instance      19.011    31.803   0.598      0.930
A   S->M        0  instance      17.856    12.117   1.474      0.994
A   S->U        0  instance      18.449    13.974   1.320      0.981
A   U->M        0  instance      17.595    33.181   0.530      0.851
A   U->S        0  instance      31.137    25.847   1.205      0.894
B   M->S        0       bag      46.951    39.948   1.175      0.847
B   M->U        0       bag      36.729    38.205   0.961      0.890
B   S->M        0       bag      34.710    29.649   1.171      0.986
B   S->U        0       bag      37.978    35.615   1.066      0.960
B   U->M        0       bag      22.070    38.125   0.579      0.792
B   U->S        0       bag      34.522    30.323   1.138      0.890
D   M->S        0  instance      29.423    24.154   1.218      0.970
D   M->U        0  instance      2

In [4]:
# Each adapted arm against its own floor. A negative change in the ratio is the
# shape conditional alignment has: the domains came together, class by class,
# more than the classes came together with each other.
compared = latent.against_floor(readings)
print(f"{'arm':<4}{'floor':<7}{'transfer':<8}{'seed':>5}"
      f"{'ratio':>9}{'floor':>9}{'change':>9}{'separable':>11}{'change':>9}")
for row in compared:
    print(f"{row['arm']:<4}{row['floor']:<7}{row['transfer']:<8}{row['seed']:>5}"
          f"{row['ratio']:>9.3f}{row['floorRatio']:>9.3f}{row['ratioChange']:>+9.3f}"
          f"{row['separability']:>11.3f}{row['separabilityChange']:>+9.3f}")

arm floor  transfer seed    ratio    floor   change  separable   change
D   A      M->S        0    1.218    1.205   +0.013      0.970   +0.023
D   A      M->U        0    0.691    0.598   +0.093      0.942   +0.012
D   A      S->M        0    1.780    1.474   +0.307      0.994   +0.001
D   A      S->U        0    1.451    1.320   +0.131      0.986   +0.005
D   A      U->M        0    0.672    0.530   +0.142      0.901   +0.050
D   A      U->S        0    1.244    1.205   +0.039      0.954   +0.060
G   B      M->S        0    1.175    1.175   +0.000      0.861   +0.014
G   B      M->U        0    0.884    0.961   -0.077      0.890   +0.000
G   B      S->M        0    1.175    1.171   +0.004      1.000   +0.014
G   B      S->U        0    1.087    1.066   +0.021      1.000   +0.040
G   B      U->M        0    0.445    0.579   -0.134      0.849   +0.056
G   B      U->S        0    1.231    1.138   +0.092      0.903   +0.013


In [5]:
# The local term's own claim, as a number: how much of the correspondence mass a
# target subject puts on source subjects of its true class. Chance is 0.100, and
# the true target labels are read here and nowhere in training.
for reading in readings:
    if "correspondence" not in reading:
        continue
    mass = reading["correspondence"]
    print(f"{reading['arm']:>2} {reading['transfer']:<6} seed {reading['seed']:<3} "
          f"mass on the true class {mass['massOnTrueClass']:.3f} "
          f"({mass['massOnTrueClass'] / mass['chance']:.1f}x chance, "
          f"{mass['subjects']} subjects)")

print()
for reading in readings:
    if "attentionSpread" not in reading:
        continue
    print(f"{reading['arm']:>2} {reading['transfer']:<6} seed {reading['seed']:<3} "
          f"attention entropy {reading['attentionSpread']:.3f} "
          f"(1.000 would be the uniform mean it is meant to improve on)")

 G M->S   seed 0   mass on the true class 0.189 (1.9x chance, 36 subjects)
 G M->U   seed 0   mass on the true class 0.382 (3.8x chance, 36 subjects)
 G S->M   seed 0   mass on the true class 0.286 (2.9x chance, 36 subjects)
 G S->U   seed 0   mass on the true class 0.206 (2.1x chance, 36 subjects)
 G U->M   seed 0   mass on the true class 0.927 (9.3x chance, 36 subjects)
 G U->S   seed 0   mass on the true class 0.135 (1.3x chance, 36 subjects)

 B M->S   seed 0   attention entropy 0.609 (1.000 would be the uniform mean it is meant to improve on)
 B M->U   seed 0   attention entropy 0.710 (1.000 would be the uniform mean it is meant to improve on)
 B S->M   seed 0   attention entropy 0.704 (1.000 would be the uniform mean it is meant to improve on)
 B S->U   seed 0   attention entropy 0.672 (1.000 would be the uniform mean it is meant to improve on)
 B U->M   seed 0   attention entropy 0.768 (1.000 would be the uniform mean it is meant to improve on)
 B U->S   seed 0   attention entro

In [6]:
# One projection per checkpoint, source as circles and target as triangles,
# coloured by class. Beside the numbers, never instead of them.
#
# UMAP and not t-SNE, and for a reason about this claim rather than about taste:
# t-SNE optimizes local neighbourhoods and does not preserve distance between
# clusters, and what the table above asserts *is* an inter-cluster distance — the
# same class across domains against different classes within one. Drawing that on a
# projection that scrambles it would show something the numbers do not say.
figures = []
for record, reading in zip(found, readings):
    model, source, target = latent.load(record, device)
    source_rows, source_labels = latent.represent(model, source, source.eval_idx, device)
    target_rows, target_labels = latent.represent(model, target, target.eval_idx, device)
    rows = torch.cat([source_rows, target_rows])
    labels = torch.cat([source_labels, target_labels])
    domains = torch.cat([torch.zeros(len(source_rows)), torch.ones(len(target_rows))])
    stem = f"{record['arm']}_{record['transfer'].replace('->', '-')}_seed{record['seed']}"
    geometry = reading["geometry"]
    figures.append(latent.projection(
        rows, labels, domains,
        config.RESULTS / "latent" / f"{stem}.png",
        f"{record['arm']} · {record['transfer']} · seed {record['seed']}",
        record["seed"],
        # The bounds travel with the picture. A figure read without them gets
        # misquoted exactly the way a number does.
        caption=(f"UMAP · {config.BAGS_PER_DOMAIN}x{config.INSTANCES_PER_BAG} bags · "
                 f"{config.EPOCHS} epochs · {len(config.SEEDS)} seed(s) · "
                 f"{config.REVISION} · ratio {geometry['ratio']:.3f} "
                 f"(same class {geometry['crossDomainSameClass']:.1f} / "
                 f"between {geometry['betweenClasses']:.1f}) · "
                 f"separable {reading['domainSeparability']:.3f}"),
    ))
    del model, source, target
print(f"{len(figures)} projections written")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


24 projections written


In [7]:
output = config.RESULTS / "latent.json"
output.write_text(json.dumps({
    "readings": readings,
    "againstFloor": compared,
    "figures": [str(path.relative_to(REPOSITORY)) for path in figures],
    "note": ("Ratios below one mean the domains sit closer to each other, class "
             "by class, than different classes sit to one another. Read the ratio "
             "and the two distances together: a fall in both is a collapse, not "
             "an alignment."),
}, indent=2), encoding="utf-8")
print("written", output.relative_to(REPOSITORY))

written MIL-CREDA/Results/Benchmark/latent.json
